# Validator Agent Notebook

This notebook demonstrates the Validator Agent which validates quantum circuits and uses LLM to fix broken code.

## Features
- **Remote Validation (optional)**: Uses QCanvas backend API for code execution (only when `agents.validator.mode == "remote"`)
- **LLM-Powered Fixing**: When code fails, the LLM analyzes errors and suggests fixes
- **Local Fallback**: Can also run in local mode using Cirq directly

## Prerequisites
1. QCanvas backend running at `http://localhost:8000` (only if `agents.validator.mode` is `remote`)
2. Ollama with `cirq-validator-agent` model created

---

# Part 1: Setup & Backend Connection Test

Before starting validation, we need to verify the backend is accessible.

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.cirq_rag_code_assistant.config import get_config, setup_logging
from src.agents.validator import ValidatorAgent
from src.tools.qcanvas_client import QCanvasClient

# Setup logging
setup_logging()
print("✅ Imports completed successfully.")

# Decide whether to run QCanvas backend tests.
# If validator.mode is not "remote", this notebook should NOT attempt QCanvas calls.
_config = get_config()
validator_mode = _config.get("agents", {}).get("validator", {}).get("mode", "local")
run_qcanvas_tests = str(validator_mode).lower() == "remote"
print(f"Validator mode: {validator_mode}")
if not run_qcanvas_tests:
    print("Skipping QCanvas backend tests (validator.mode != 'remote').")

2026-03-26 11:38:39 | INFO     | src.cirq_rag_code_assistant.config.logging:setup_all:127 | Logging configuration completed
2026-03-26 11:38:39 | INFO     | config.config_loader:load:101 | ✅ Loaded configuration from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\config\config.json


✅ Imports completed successfully.
Validator mode: local
Skipping QCanvas backend tests (validator.mode != 'remote').


## 1.1 Check Backend Health

Verify the QCanvas backend is running and accessible.

In [2]:
# Initialize QCanvas client (only when running in remote mode)
run_qcanvas_tests = globals().get("run_qcanvas_tests", False)
client = globals().get("client", None)

if run_qcanvas_tests:
    client = QCanvasClient()
    print(f"Backend URL: {client.base_url}")
    print(f"Timeout: {client.timeout}s")
    print()

    # Check health
    backend_healthy = client.check_health()

    if backend_healthy:
        print("✅ QCanvas backend is HEALTHY!")
        frameworks = client.get_supported_frameworks()
        print(f"   Supported frameworks: {frameworks}")
    else:
        print("❌ QCanvas backend is NOT AVAILABLE!")
        print("")
        print("Please start the backend:")
        print("  cd ../backend")
        print("  python start.py")
else:
    client = None
    backend_healthy = False
    print("Skipping QCanvas backend health check (validator.mode != 'remote').")

Skipping QCanvas backend health check (validator.mode != 'remote').


## 1.2 Test Backend API: Convert Code to QASM

Test the `/api/converter/convert` endpoint with sample Cirq code.

In [3]:
# Test code for conversion
test_code = """
import cirq

# Simple Bell state circuit
q0, q1 = cirq.LineQubit.range(2)
circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key='m')
)
"""

run_qcanvas_tests = globals().get("run_qcanvas_tests", False)
client = globals().get("client", None)

if run_qcanvas_tests and client is not None:
    print("📤 Sending code to /api/converter/convert...")
    print()
    conv_result = client.convert_to_qasm(test_code, framework="cirq")
else:
    print("⚠️ Skipping /api/converter/convert (validator.mode != 'remote').")
    conv_result = {"success": False, "error": "QCanvas conversion skipped"}

print(f"Success: {conv_result.get('success')}")
print()

if conv_result.get('success'):
    print("✅ Conversion successful!")
    print()
    print("📄 Generated OpenQASM 3.0:")
    print("-" * 40)
    qasm_code = conv_result.get('qasm_code', '')
    print(qasm_code)
    print("-" * 40)
    
    stats = conv_result.get('conversion_stats', {})
    if stats:
        print(f"\nConversion Stats: {stats}")
else:
    print("❌ Conversion failed!")
    print(f"Error: {conv_result.get('error')}")

⚠️ Skipping /api/converter/convert (validator.mode != 'remote').
Success: False

❌ Conversion failed!
Error: QCanvas conversion skipped


## 1.3 Test Backend API: Execute QASM Simulation

Test the `/api/simulator/execute-qsim` endpoint with the generated QASM.

In [4]:
# Only run if conversion was successful
run_qcanvas_tests = globals().get("run_qcanvas_tests", False)
client = globals().get("client", None)

if run_qcanvas_tests and client is not None and conv_result.get('success') and conv_result.get('qasm_code'):
    qasm_code = conv_result['qasm_code']
    
    print("📤 Sending QASM to /api/simulator/execute-qsim...")
    print(f"   Backend: cirq")
    print(f"   Shots: 1024")
    print()
    
    sim_result = client.execute_qasm(qasm_code, backend="cirq", shots=1024)
    
    print(f"Success: {sim_result.get('success')}")
    print()
    
    if sim_result.get('success'):
        print("✅ Simulation successful!")
        print()
        
        results = sim_result.get('results', {})
        
        print("📊 SIMULATION RESULTS:")
        print("-" * 40)
        print(f"Counts: {results.get('counts', {})}")
        print(f"Probabilities: {results.get('probs', {})}")
        print()
        
        metadata = results.get('metadata', {})
        if metadata:
            print("📈 Metadata:")
            print(f"   Execution Time: {metadata.get('execution_time', 'N/A')}")
            print(f"   Simulation Time: {metadata.get('simulation_time', 'N/A')}")
            print(f"   N Qubits: {metadata.get('n_qubits', 'N/A')}")
            print(f"   Fidelity: {metadata.get('fidelity', 'N/A')}%")
            print(f"   Memory Usage: {metadata.get('memory_usage', 'N/A')}")
    else:
        print("❌ Simulation failed!")
        print(f"Error: {sim_result.get('error')}")
else:
    print("⚠️ Skipping simulation test - conversion was not successful.")

⚠️ Skipping simulation test - conversion was not successful.


## 1.4 Test Full Pipeline: validate_and_execute()

Test the combined convert + execute pipeline.

In [5]:
print("📤 Testing full pipeline (convert + execute)...")
print()

run_qcanvas_tests = globals().get("run_qcanvas_tests", False)
client = globals().get("client", None)

if run_qcanvas_tests and client is not None:
    full_result = client.validate_and_execute(test_code, shots=1024)
else:
    full_result = {
        "success": False,
        "stage": "skipped",
        "error": "QCanvas full pipeline skipped (validator.mode != 'remote')"
    }

print(f"Success: {full_result.get('success')}")
print(f"Stage: {full_result.get('stage')}")
print()

if full_result.get('success'):
    print("✅ Full pipeline successful!")
    results = full_result.get('results', {})
    print(f"   Counts: {results.get('counts', {})}")
else:
    print("❌ Pipeline failed at stage:", full_result.get('stage'))
    print(f"   Error: {full_result.get('error')}")

print()
print("=" * 50)
print("🎉 BACKEND TEST COMPLETE - Ready for Validation!")
print("=" * 50)

📤 Testing full pipeline (convert + execute)...

Success: False
Stage: skipped

❌ Pipeline failed at stage: skipped
   Error: QCanvas full pipeline skipped (validator.mode != 'remote')

🎉 BACKEND TEST COMPLETE - Ready for Validation!


---

# Part 2: Validator Agent Testing

Now that the backend is verified, let's test the ValidatorAgent.

## 2.1 Initialize Validator Agent

In [6]:
# First, add imports and setup RAG components
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever

# Initialize Knowledge Base with all datasets
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
kb.load_from_directory()
kb.index_entries()
print(f"Loaded {len(kb.entries)} KB entries for semantic validation")

# Create retriever
retriever = Retriever(knowledge_base=kb)

# Initialize ValidatorAgent with RAG retriever (mode comes from config.json)
validator = ValidatorAgent(retriever=retriever)  # Mode is read from config.json

print("✅ ValidatorAgent initialized with RAG support")
...
print(f"   Mode: {validator.mode}")
print(f"   LLM Enabled: {validator.llm_enabled}")
print(f"   Ollama Model: {validator.ollama_model}")
print(f"   Default Shots: {validator.default_shots}")


2026-03-26 11:38:39,491 - botocore.credentials - INFO - credentials.py:1252 - Found credentials in environment variables.


2026-03-26 11:38:39 | INFO     | src.rag.embeddings:_init_aws:124 | Using AWS Bedrock Embeddings: amazon.nova-2-multimodal-embeddings-v1:0, dim=1024
2026-03-26 11:38:39 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2026-03-26 11:38:39 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2026-03-26 11:38:39 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 1024
2026-03-26 11:38:39 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2026-03-26 11:38:39 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.jsonl
2026-03-26 11:38:39 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 107 entries from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.js

Loaded 140 KB entries for semantic validation
✅ ValidatorAgent initialized with RAG support
   Mode: local
   LLM Enabled: True
   Ollama Model: arn:aws:bedrock:us-east-1:626230557862:application-inference-profile/lupvf1p1uqzg
   Default Shots: 1024


## 2.2 Validate Valid Code (Bell State)

In [7]:
valid_code = """
import cirq

# Create a Bell state circuit
q0, q1 = cirq.LineQubit.range(2)
circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key='result')
)
print(circuit)
"""

task = {
    "code": valid_code,
    "description": "Create a Bell state with two entangled qubits",
    "shots": 1024
}

print("🔍 Validating Bell state code...")
result = validator.execute(task)

print(f"\n{'✅' if result.get('success') else '❌'} Validation {'PASSED' if result.get('success') else 'FAILED'}")
print(f"Stage: {result.get('stage')}")

if result.get('success'):
    results = result.get('results', {})
    print(f"\n📊 Results:")
    print(f"   Counts: {results.get('counts', {})}")
    print(f"   Probabilities: {results.get('probs', {})}")
else:
    print(f"\n🔴 Error: {result.get('error')}")

2026-03-26 11:51:19 | INFO     | src.agents.validator:execute:157 | Validating code (attempt 1/3)...


🔍 Validating Bell state code...


2026-03-26 11:51:19 | INFO     | src.tools.simulator:simulate:133 | Simulation completed. Histogram: {'00': 509, '11': 515}
2026-03-26 11:51:20 | INFO     | src.agents.validator:execute:185 | Validation passed on attempt 1



✅ Validation PASSED
Stage: simulation

📊 Results:
   Counts: {'00': 509, '11': 515}
   Probabilities: None


## 2.3 Validate Invalid Code (with LLM Fix)

In [8]:
# Code with missing import and missing measurement
invalid_code = """
# Missing import cirq!
q0, q1 = cirq.LineQubit.range(2)
circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1)
)
"""

task = {
    "code": invalid_code,
    "description": "Create a simple quantum circuit with H and CNOT gates",
    "force_llm_fix": True
}

print("🔍 Validating broken code (missing import)...")
result = validator.execute(task)

print(f"\n{'✅' if result.get('success') else '❌'} Validation {'PASSED' if result.get('success') else 'FAILED'}")
print(f"Stage: {result.get('stage')}")

if result.get('error'):
    print(f"\n🔴 Error: {result.get('error')}")

# Show LLM analysis
llm_analysis = result.get('llm_analysis', {})
if llm_analysis:
    print("\n🤖 LLM Analysis:")
    if llm_analysis.get('fixed_code'):
        print("\n📝 Fixed Code:")
        print("```python")
        print(llm_analysis['fixed_code'])
        print("```")
    if llm_analysis.get('analysis'):
        print(f"\n💡 Explanation: {llm_analysis['analysis']}")

2026-03-26 11:51:20 | INFO     | src.agents.validator:execute:157 | Validating code (attempt 1/3)...
2026-03-26 11:51:20 | INFO     | src.agents.validator:_execute_local:535 | Running LLM analysis for missing measurement fixing...


🔍 Validating broken code (missing import)...


2026-03-26 11:51:25 | INFO     | src.agents.validator:execute:192 | Using LLM-fixed code for retry...
2026-03-26 11:51:25 | INFO     | src.agents.validator:execute:159 | Re-validating fixed code (attempt 2/3)...
2026-03-26 11:51:25 | INFO     | src.tools.simulator:simulate:133 | Simulation completed. Histogram: {'11': 510, '00': 514}
2026-03-26 11:51:25 | INFO     | src.agents.validator:_execute_local:602 | Running LLM analysis for code fixing...
2026-03-26 11:51:33 | INFO     | src.agents.validator:execute:185 | Validation passed on attempt 2



✅ Validation PASSED
Stage: simulation

🤖 LLM Analysis:

📝 Fixed Code:
```python
import cirq

q0, q1 = cirq.LineQubit.range(2)
circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key="m")
)

print(circuit)

simulator = cirq.Simulator()
result = simulator.run(circuit, repetitions=1000)
print("Measurement results:", result)

# Display counts
counts = result.measurements["m"]
from collections import Counter
bitstring_counts = Counter(
    "".join(str(b) for b in row) for row in counts
)
print("Counts:", dict(bitstring_counts))
```

💡 Explanation: The original code executed successfully without errors. The simulation results show the expected Bell state behavior with roughly equal counts of '00' and '11' (510 and 514 respectively out of 1024 total), which is correct for a Bell state circuit. The only minor adjustment made was increasing repetitions from 100 to 1000 for better statistical representation, and adding explicit count display using `Counter` t

## 2.4 Re-validate Fixed Code

In [9]:
fixed_code = result.get('fixed_code') or result.get('llm_analysis', {}).get('fixed_code')

if fixed_code:
    print("🔍 Re-validating LLM-fixed code...")
    
    task_fixed = {
        "code": fixed_code,
        "description": "Fixed version of the circuit",
        "shots": 1024
    }
    
    result_fixed = validator.execute(task_fixed)
    
    print(f"\n{'✅' if result_fixed.get('success') else '❌'} Fixed code {'PASSED' if result_fixed.get('success') else 'FAILED'}")
    
    if result_fixed.get('success'):
        results = result_fixed.get('results', {})
        print(f"   Counts: {results.get('counts', {})}")
    else:
        print(f"   Error: {result_fixed.get('error')}")
else:
    print("⚠️ No fixed code available from previous step.")

2026-03-26 11:51:33 | INFO     | src.agents.validator:execute:157 | Validating code (attempt 1/3)...
2026-03-26 11:51:33 | INFO     | src.tools.simulator:simulate:133 | Simulation completed. Histogram: {'00': 543, '11': 481}


🔍 Re-validating LLM-fixed code...


2026-03-26 11:51:33 | INFO     | src.agents.validator:execute:185 | Validation passed on attempt 1



✅ Fixed code PASSED
   Counts: {'00': 543, '11': 481}


---
# Part 3: Comprehensive Test Suite

Run the full suite of 20+ test cases to verify the Validator's robustness.

In [10]:
from tests.test_validator_circuits import run_tests

# Run all test cases
df_results = run_tests(validator)

# Display summary
print("\n" + "="*40)
print("TEST SUITE SUMMARY")
print("="*40)
print(f"Total Tests: {len(df_results)}")
print(f"Passed: {len(df_results[df_results['Status'].str.contains('PASS')])}")
print(f"Failed: {len(df_results[df_results['Status'].str.contains('FAIL|ERROR')])}")

# Show full table
df_results

ImportError: cannot import name 'run_tests' from 'tests.test_validator_circuits' (D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\tests\test_validator_circuits.py)